In [3]:
import torch
import torch.nn as nn
import math
import time
import numpy as np
import pandas as pd
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
import pandas as pd
import numpy as np
import os
import json

train_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/train.parquet"

cat_cols = [f"C{i}" for i in range(1, 27)]

start_time = time.time()
print("Tính Global CTR...")
df = pd.read_parquet(train_file)

total_clicks = df["label"].sum()
total_impressions = len(df)
ctr_global = total_clicks / total_impressions if total_impressions > 0 else 0

ctr_time = time.time()
print(f"Global CTR: {ctr_global:.4f}")
print(f"Thời gian chạy Global CTR: {(ctr_time - start_time)} giây.")

print("Gom nhóm và đếm đếm tần suất ID...")
# Melt (Unpivot) 26 cột thành 1 cột
df_melted = df.melt(id_vars=["label"], value_vars=cat_cols, var_name="Feature_Name", value_name="Raw_Value")
df_melted = df_melted.dropna(subset=["Raw_Value"])
df_melted["Global_ID"] = df_melted["Feature_Name"] + "_" + df_melted["Raw_Value"].astype(str)

agg = df_melted.groupby("Global_ID").agg(N=("label", "count"), Clicks=("label", "sum")).reset_index()

spark_time = time.time()
print(f"Hoàn tất! Kích thước từ điển: {len(agg):,} IDs.")
print(f"Thời gian chạy gom nhóm và đếm tần suất ID: {(spark_time - start_time)} giây")

print("Tính IVS và chia embedding table...")
# Tính IVS 
agg["ctr_id"] = np.where(agg["N"] > 0, agg["Clicks"] / agg["N"], 0)
agg["IVS"] = np.log10(agg["N"] + 1) * np.abs(agg["ctr_id"] - ctr_global)

# Sắp xếp và tính toán tích lũy
df_ivs = agg.sort_values("IVS", ascending=False).copy()
total_ivs = df_ivs["IVS"].sum()

df_ivs["Cum_Percent"] = df_ivs["IVS"].cumsum() / total_ivs if total_ivs > 0 else 0

def assign_group(p):
    if p <= 0.50: return 64
    elif p <= 0.70: return 32
    elif p <= 0.85: return 16
    elif p <= 0.95: return 8
    elif p <= 0.99: return 4
    else: return 2

df_ivs["Dim_Group"] = df_ivs["Cum_Percent"].apply(assign_group)

# Tạo Local Index (Bắt đầu từ 1, chừa 0 cho Padding)
df_ivs["Local_Index"] = df_ivs.groupby("Dim_Group").cumcount() + 1

mapping_dict = df_ivs.set_index("Global_ID")[["Dim_Group", "Local_Index"]].to_dict('index')
vocab_sizes = df_ivs.groupby("Dim_Group").size().to_dict()
vocab_sizes = {str(k): int(v) for k, v in vocab_sizes.items()}

with open("tame_ivs_dictionary.json", "w") as f:
    json.dump(mapping_dict, f)

with open("tame_vocab_sizes.json", "w") as f:
    json.dump(vocab_sizes, f)


total_time = time.time()
print(f"Hoàn tất! Tổng số ID: {len(mapping_dict)}")
print(f"Kích thước từ vựng các nhóm: {vocab_sizes}")
print(f"Tổng thời gian: {(total_time - start_time)} giây.")

Tính Global CTR...
Global CTR: 0.2562
Thời gian chạy Global CTR: 47.385480642318726 giây.
Gom nhóm và đếm đếm tần suất ID...
Hoàn tất! Kích thước từ điển: 28,222,759 IDs.
Thời gian chạy gom nhóm và đếm tần suất ID: 306.5376043319702 giây
Tính IVS và chia embedding table...
Hoàn tất! Tổng số ID: 28222759
Kích thước từ vựng các nhóm: {'2': 897072, '4': 1832151, '8': 4580378, '16': 6870568, '32': 6794660, '64': 7247930}
Tổng thời gian: 404.22159028053284 giây.


In [1]:
import os
import psutil
from pyspark.sql import SparkSession

cpu_cores = os.cpu_count()
total_ram_gb = psutil.virtual_memory().total / (1024**3)

print("Thông số máy:")
print(f"CPU (Cores): {cpu_cores}")
print(f"RAM: {total_ram_gb:.2f} GB")

spark_memory_gb = int(total_ram_gb * 0.85)

shuffle_partitions = cpu_cores * 3

print("Cấu hình Spark:")
print(f"- spark.driver.memory: {spark_memory_gb}g")
print(f"- spark.sql.shuffle.partitions: {shuffle_partitions}")

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Criteo_TAME_Preprocess") \
    .config("spark.driver.memory", f"{spark_memory_gb}g") \
    .config("spark.sql.shuffle.partitions", str(shuffle_partitions)) \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.driver.maxResultSize", "10g") \
    .getOrCreate()

print("Khởi tạo Spark thành công!")

Thông số máy:
CPU (Cores): 48
RAM: 176.88 GB
Cấu hình Spark:
- spark.driver.memory: 150g
- spark.sql.shuffle.partitions: 144


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 04:31:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Khởi tạo Spark thành công!


In [4]:
import glob
import os
import json
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

train_file = "/kaggle/input/datasets/nmpogg/criteo-cleaned-v2/train.parquet"
cat_cols = [f"C{i}" for i in range(1, 27)]

start_time = time.time()

print("Tính Global CTR...")
df = spark.read.parquet(train_file)
stats_row = df.select(
    F.sum("label").alias("total_clicks"), 
    F.count("*").alias("total_impressions")
).collect()[0]

ctr_global = stats_row["total_clicks"] / stats_row["total_impressions"]

ctr_time = time.time()
print(f"Global CTR: {ctr_global:.4f}")
print(f"Thời gian chạy Global CTR: {(ctr_time - start_time)} giây.")

print("Gom nhóm và đếm đếm tần suất ID...")

unpivot_expr = f"stack({len(cat_cols)}, " + ", ".join([f"'{c}', {c}" for c in cat_cols]) + ") as (Feature_Name, Raw_Value)"

agg_df = df.select("label", F.expr(unpivot_expr)) \
    .filter(F.col("Raw_Value").isNotNull()) \
    .withColumn("Global_ID", F.concat_ws("_", F.col("Feature_Name"), F.col("Raw_Value"))) \
    .groupBy("Global_ID") \
    .agg(
        F.count("label").alias("N"),
        F.sum("label").alias("Clicks")
    )

final_stats = agg_df.toPandas()
final_stats.set_index("Global_ID", inplace=True)

spark_time = time.time()
print(f"Hoàn tất! Kích thước từ điển: {len(final_stats):,} IDs.")
print(f"Thời gian chạy gom nhóm và đếm tần suất ID: {(spark_time - start_time)} giây")

print("Tính IVS và chia embedding table...")
final_stats["CTR_ID"] = final_stats["Clicks"] / final_stats["N"]
final_stats["IVS"] = np.log10(final_stats["N"] + 1) * np.abs(final_stats["CTR_ID"] - ctr_global)

df_ivs = final_stats.sort_values("IVS", ascending=False).reset_index()
total_ivs = df_ivs["IVS"].sum()
df_ivs["Cum_Percent"] = df_ivs["IVS"].cumsum() / total_ivs

def assign_group(p):
    if p <= 0.50: return 64
    elif p <= 0.70: return 32
    elif p <= 0.85: return 16
    elif p <= 0.95: return 8
    elif p <= 0.99: return 4
    else: return 2

df_ivs["Dim_Group"] = df_ivs["Cum_Percent"].apply(assign_group)
df_ivs["Local_Index"] = df_ivs.groupby("Dim_Group").cumcount() + 1

mapping_dict = df_ivs.set_index("Global_ID")[["Dim_Group", "Local_Index"]].to_dict('index')
vocab_sizes = df_ivs.groupby("Dim_Group").size().to_dict()
vocab_sizes = {str(k): int(v) for k, v in vocab_sizes.items()}

with open("tame_ivs_dictionary.json", "w") as f:
    json.dump(mapping_dict, f)

with open("tame_vocab_sizes.json", "w") as f:
    json.dump(vocab_sizes, f)

total_time = time.time()
print(f"Hoàn tất! Tổng số ID: {len(mapping_dict)}")
print(f"Kích thước từ vựng các nhóm: {vocab_sizes}")
print(f"Tổng thời gian: {(total_time - start_time)} giây.")

Tính Global CTR...


26/05/25 04:31:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Global CTR: 0.2562
Thời gian chạy Global CTR: 13.64600920677185 giây.
Gom nhóm và đếm đếm tần suất ID...


Hoàn tất! Kích thước từ điển: 28,222,759 IDs.
Thời gian chạy gom nhóm và đếm tần suất ID: 39.873106718063354 giây
Tính IVS và chia embedding table...
Hoàn tất! Tổng số ID: 28222759
Kích thước từ vựng các nhóm: {'2': 897072, '4': 1832151, '8': 4580378, '16': 6870568, '32': 6794660, '64': 7247930}
Tổng thời gian: 128.27837443351746 giây.
